# RNN → Transformer 古诗生成 V3

以 **"明月"** 为起始词，基于 **GPT 风格 Transformer** 生成七言绝句。

**V3 改进点（相对 V2）**
- 用 Transformer Decoder（GPT 风格）替换 LSTM
- 可学习位置编码（Learnable Positional Encoding）
- Multi-Head Self-Attention + 因果掩码
- FFN 激活函数可选：`relu` / `gelu` / `tanh`
- 多层 FC 解码头 + 残差连接（与 V2 一致，方便对比）
- 同款训练策略：Warmup + CosineAnnealing + Label Smoothing
- 同款生成策略：Temperature / Top-k / Top-p 三选一

**文件结构**
```
homework3/
├── rnn_poem_generation_v3.ipynb   ← 本文件
└── data_files/                    ← 所有 JSON 数据
    ├── poet_song_40000.json
    └── ...
```

## 0. 安装依赖（首次运行取消注释）

In [1]:
# !pip install torch matplotlib

## 1. 导入库 & 全局配置

In [2]:
import json, os, re, random, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ════════════════════════════════════════════════════
# 全局配置
# ════════════════════════════════════════════════════
CONFIG = {
    # ── 数据 ──────────────────────────────────────────
    "data_files": [f for f in os.listdir("data_files") if f.endswith(".json")],
    "data_dir":   "data_files",

    # ── 模型结构 ──────────────────────────────────────
    "seq_len":       32,        # 七言绝句固定序列长度（含标点）
    "d_model":       256,       # Transformer 隐层维度（embedding 维度）
    "nhead":         8,         # Multi-Head Attention 头数
    "num_layers":    4,         # Transformer Decoder 层数
    "d_ffn":         512,       # FFN 中间层维度（通常 = 2*d_model）
    "dropout":       0.1,       # Transformer 内部 dropout
    "activation":    "gelu",    # FFN 激活函数: "relu" / "gelu" / "tanh"

    # ── 训练超参 ──────────────────────────────────────
    "batch_size":      128,
    "num_epochs":      40,
    "learning_rate":   3e-4,
    "warmup_epochs":   4,
    "label_smoothing": 0.1,
    "clip_grad":       1.0,
    "seed":            42,

    # ── 生成参数 ──────────────────────────────────────
    "start_words":  "明月",
    "sample_mode":  "topp",     # "temperature" / "topk" / "topp"
    "temperature":  0.8,
    "top_k":        10,
    "top_p":        0.9,

    # ── 输出 ──────────────────────────────────────────
    "save_model": "poem_transformer_v3.pth",
    "loss_fig":   "training_loss_v3.png",
}

# ── 随机种子 & 设备 ──────────────────────────────────
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"数据文件数量: {len(CONFIG['data_files'])}")

PAD, START, END, UNK = "<PAD>", "<START>", "<END>", "<UNK>"


Using device: cuda
数据文件数量: 317


## 2. 数据加载与预处理

In [3]:
_QIYAN_PATTERN = re.compile(
    r'^[\u4e00-\u9fff]{7}[，,][\u4e00-\u9fff]{7}[。！？]$'
)

def is_qiyan_jueju(paragraphs):
    """判断是否为七言绝句（2联，每联16字）"""
    if len(paragraphs) != 2:
        return False
    for p in paragraphs:
        ps = p.strip()
        if len(ps) != 16 or not _QIYAN_PATTERN.match(ps):
            return False
    return True


def load_sequences(data_dir, filenames):
    """过滤七言绝句，展开为 32 字符序列"""
    sequences = []
    for fname in filenames:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"[warning] 跳过: {fpath}")
            continue
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            para = item.get("paragraphs", [])
            if is_qiyan_jueju(para):
                seq = "".join(p.strip() for p in para)
                # 二次校验：过滤非汉字/标点杂质
                seq_clean = re.sub(r'[^\u4e00-\u9fff，。！？]', '', seq)
                if len(seq_clean) == 32:
                    sequences.append(seq_clean)

    print(f"过滤后七言绝句: {len(sequences)} 首")
    assert len(sequences) > 0, "未找到七言绝句！请检查 data_dir 配置。"
    return sequences


def build_vocab(sequences):
    """构建字符级词表"""
    chars = sorted(set("".join(sequences)))
    vocab = [PAD, START, END, UNK] + chars
    char2idx = {c: i for i, c in enumerate(vocab)}
    idx2char = {i: c for c, i in char2idx.items()}
    print(f"词表大小: {len(vocab)}")
    return char2idx, idx2char, vocab


sequences            = load_sequences(CONFIG["data_dir"], CONFIG["data_files"])
char2idx, idx2char, vocab = build_vocab(sequences)

print("\n示例序列：")
for s in sequences[:3]:
    print(" ", s)


过滤后七言绝句: 82949 首
词表大小: 8048

示例序列：
  欲出未出光辣達，千山萬山如火發。須臾走向天上來，逐却殘星趕却月。
  片片飛來靜又閑，樓頭江上復山前。飄零盡日不歸去，帖破清光萬里天。
  一氣東南王斗牛，祖龍潜爲子孫憂。金陵地脈何曾斷，不覺真人已姓劉。


## 3. Dataset

In [4]:
class PoemDataset(Dataset):
    """
    Teacher Forcing 样本对，序列长度 32：
        inp = [START] + seq[:-1]
        tgt = seq
    """
    def __init__(self, sequences, char2idx):
        start_id = char2idx[START]
        unk_id   = char2idx[UNK]
        self.data = []
        for seq in sequences:
            ids = [char2idx.get(c, unk_id) for c in seq]
            self.data.append((
                torch.tensor([start_id] + ids[:-1], dtype=torch.long),
                torch.tensor(ids, dtype=torch.long),
            ))

    def __len__(self):         return len(self.data)
    def __getitem__(self, i):  return self.data[i]


dataset = PoemDataset(sequences, char2idx)
print(f"训练样本数: {len(dataset)}")
inp0, tgt0 = dataset[0]
print(f"inp shape: {inp0.shape},  tgt shape: {tgt0.shape}")


训练样本数: 82949
inp shape: torch.Size([32]),  tgt shape: torch.Size([32])


## 4. Transformer 模型定义

### 结构图
```
输入 token ids (B, L)
    ↓
Embedding(vocab_size, d_model)          # 字符嵌入
    ↓
+ LearnablePosEncoding(max_len, d_model) # 可学习位置编码
    ↓  Dropout
TransformerDecoderLayer × num_layers    # 每层：
    ├─ Masked Multi-Head Self-Attention  #   因果自注意力
    ├─ Add & LayerNorm
    ├─ FFN: Linear→Activation→Dropout→Linear
    └─ Add & LayerNorm
    ↓
FC(d_model → d_model//2) + Activation   # 多层解码头
    ↓  + Residual
FC(d_model//2 → vocab_size)
```

> **因果掩码**：确保位置 i 只能看到位置 ≤ i 的字符，与 GPT 原理相同。


In [5]:
def get_activation(name: str) -> nn.Module:
    """根据名称返回激活函数，支持 relu / gelu / tanh"""
    mapping = {"relu": nn.ReLU(), "gelu": nn.GELU(), "tanh": nn.Tanh()}
    name = name.lower()
    if name not in mapping:
        raise ValueError(f"不支持的激活函数: {name}，可选: {list(mapping)}")
    return mapping[name]


class LearnablePosEncoding(nn.Module):
    """
    可学习位置编码。
    与固定 sin/cos 编码相比，更适合固定长度的短序列（如32字古诗）。
    """
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, d_model)"""
        B, L, _ = x.shape
        positions = torch.arange(L, device=x.device).unsqueeze(0)  # (1, L)
        return x + self.pos_emb(positions)                          # (B, L, d_model)


class PoemTransformer(nn.Module):
    """
    GPT 风格 Transformer 语言模型（仅 Decoder）

    参数：
        vocab_size  : 词表大小
        d_model     : Transformer 隐层维度
        nhead       : Multi-Head Attention 头数（d_model 须被 nhead 整除）
        num_layers  : Transformer Decoder 层数
        d_ffn       : FFN 中间层维度
        dropout     : Dropout 比率
        max_len     : 最大序列长度（含 START token，设为 seq_len+1 即可）
        activation  : FFN 激活函数名称
    """
    def __init__(self, vocab_size, d_model, nhead, num_layers,
                 d_ffn, dropout, max_len, activation="gelu"):
        super().__init__()

        assert d_model % nhead == 0,             f"d_model({d_model}) 必须能被 nhead({nhead}) 整除"

        # ── 输入层 ──────────────────────────────────────
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = LearnablePosEncoding(max_len, d_model)
        self.emb_drop  = nn.Dropout(dropout)

        # ── Transformer Decoder 层 ───────────────────────
        # 使用 PyTorch 内置 TransformerDecoderLayer
        # 由于是纯语言模型（无 Encoder），memory = 零张量（占位）
        decoder_layer = nn.TransformerDecoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = d_ffn,
            dropout         = dropout,
            activation      = activation if activation in ("relu","gelu") else "gelu",
            batch_first     = True,
            norm_first      = True,   # Pre-LN，训练更稳定
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)

        # ── 多层 FC 解码头 + 残差 ────────────────────────
        mid = d_model // 2
        self.fc1          = nn.Linear(d_model, mid)
        self.act          = get_activation(activation)
        self.fc_drop      = nn.Dropout(dropout)
        self.fc2          = nn.Linear(mid, vocab_size)
        self.residual_proj = nn.Linear(d_model, mid, bias=False)

        # ── 权重初始化 ───────────────────────────────────
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0, std=0.02)

    @staticmethod
    def _make_causal_mask(seq_len: int, device) -> torch.Tensor:
        """
        生成因果掩码（上三角矩阵），确保位置 i 只能 attend 到 ≤ i 的位置。
        True 表示被屏蔽（masked out）。
        形状: (seq_len, seq_len)
        """
        mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device), diagonal=1
        ).bool()
        return mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x      : (B, L)  整数 token id
        返回   : logits (B, L, vocab_size)
        """
        B, L = x.shape

        # 因果掩码
        causal_mask = self._make_causal_mask(L, x.device)   # (L, L)

        # Embedding + 位置编码
        emb = self.token_emb(x)           # (B, L, d_model)
        emb = self.pos_enc(emb)           # (B, L, d_model)
        emb = self.emb_drop(emb)

        # 零占位 memory（GPT 风格，无 Encoder 输出）
        memory = torch.zeros(B, 1, emb.size(-1), device=x.device)

        # Transformer Decoder（Self-Attention 使用因果掩码）
        out = self.transformer(
            tgt            = emb,
            memory         = memory,
            tgt_mask       = causal_mask,
            tgt_is_causal  = True,
        )                                  # (B, L, d_model)

        # 多层 FC 解码头 + 残差
        res    = self.residual_proj(out)                    # (B, L, mid)
        h      = self.act(self.fc1(out))                    # (B, L, mid)
        h      = self.fc_drop(h + res)                      # 残差相加
        logits = self.fc2(h)                                # (B, L, vocab_size)

        return logits


# ── 实例化 ───────────────────────────────────────────
model = PoemTransformer(
    vocab_size  = len(vocab),
    d_model     = CONFIG["d_model"],
    nhead       = CONFIG["nhead"],
    num_layers  = CONFIG["num_layers"],
    d_ffn       = CONFIG["d_ffn"],
    dropout     = CONFIG["dropout"],
    max_len     = CONFIG["seq_len"] + 2,   # 留余量
    activation  = CONFIG["activation"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {n_params:,}")
print(model)


模型参数量: 6,335,984
PoemTransformer(
  (token_emb): Embedding(8048, 256, padding_idx=0)
  (pos_enc): LearnablePosEncoding(
    (pos_emb): Embedding(34, 256)
  )
  (emb_drop): Dropout(p=0.1, inplace=False)
  (transformer): TransformerDecoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm3): LayerNorm((256,), eps=1e-05, eleme

## 5. 学习率调度器

**Warmup + CosineAnnealing**（与 V2 完全相同，方便对比）
- 前 `warmup_epochs` 个 epoch 线性升温
- 之后余弦衰减至接近 0


In [6]:
class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch
        lrs = []
        for base_lr in self.base_lrs:
            if ep < self.warmup_epochs:
                lr = base_lr * (ep + 1) / self.warmup_epochs
            else:
                progress = (ep - self.warmup_epochs) / max(
                    1, self.total_epochs - self.warmup_epochs)
                lr = base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
            lrs.append(max(lr, 1e-7))
        return lrs

print("WarmupCosineScheduler 定义完毕。")


WarmupCosineScheduler 定义完毕。


## 6. 训练

In [7]:
def train_epoch(model, loader, optimizer, criterion, device, clip):
    """训练一个 epoch，返回平均 per-token loss"""
    model.train()
    total_loss, total_n = 0.0, 0
    for inp, tgt in loader:
        inp, tgt = inp.to(device), tgt.to(device)

        logits = model(inp)                             # (B, L, V)
        loss   = criterion(
            logits.reshape(-1, logits.size(-1)),        # (B*L, V)
            tgt.reshape(-1)                             # (B*L,)
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item() * tgt.numel()
        total_n    += tgt.numel()
    return total_loss / total_n


# ── 初始化训练组件 ───────────────────────────────────
loader    = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                       shuffle=True, num_workers=0)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = CONFIG["learning_rate"],
    weight_decay = 1e-2,    # AdamW 权重衰减，Transformer 标配
)
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs = CONFIG["warmup_epochs"],
    total_epochs  = CONFIG["num_epochs"],
)
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

# ── 训练循环 ─────────────────────────────────────────
epoch_losses = []

for epoch in range(1, CONFIG["num_epochs"] + 1):
    loss = train_epoch(model, loader, optimizer, criterion,
                       DEVICE, CONFIG["clip_grad"])
    scheduler.step()
    epoch_losses.append(loss)

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
          f"Loss: {loss:.4f}  LR: {lr_now:.2e}")

print("\n训练完成！")


Epoch [01/40]  Loss: 6.6650  LR: 1.50e-04
Epoch [02/40]  Loss: 6.0656  LR: 2.25e-04
Epoch [03/40]  Loss: 5.7689  LR: 3.00e-04
Epoch [04/40]  Loss: 5.5882  LR: 3.00e-04
Epoch [05/40]  Loss: 5.4469  LR: 2.99e-04
Epoch [06/40]  Loss: 5.3503  LR: 2.98e-04
Epoch [07/40]  Loss: 5.2895  LR: 2.95e-04
Epoch [08/40]  Loss: 5.2442  LR: 2.91e-04
Epoch [09/40]  Loss: 5.2075  LR: 2.86e-04
Epoch [10/40]  Loss: 5.1759  LR: 2.80e-04
Epoch [11/40]  Loss: 5.1481  LR: 2.73e-04
Epoch [12/40]  Loss: 5.1248  LR: 2.65e-04
Epoch [13/40]  Loss: 5.1033  LR: 2.56e-04
Epoch [14/40]  Loss: 5.0840  LR: 2.46e-04
Epoch [15/40]  Loss: 5.0651  LR: 2.36e-04
Epoch [16/40]  Loss: 5.0497  LR: 2.25e-04
Epoch [17/40]  Loss: 5.0347  LR: 2.13e-04
Epoch [18/40]  Loss: 5.0196  LR: 2.01e-04
Epoch [19/40]  Loss: 5.0058  LR: 1.89e-04
Epoch [20/40]  Loss: 4.9937  LR: 1.76e-04
Epoch [21/40]  Loss: 4.9819  LR: 1.63e-04
Epoch [22/40]  Loss: 4.9710  LR: 1.50e-04
Epoch [23/40]  Loss: 4.9601  LR: 1.37e-04
Epoch [24/40]  Loss: 4.9499  LR: 1

## 7. 保存模型

In [8]:
torch.save({
    "model_state_dict": model.state_dict(),
    "char2idx": char2idx,
    "idx2char":  idx2char,
    "vocab":     vocab,
    "config":    CONFIG,
}, CONFIG["save_model"])
print(f"模型已保存 → {CONFIG['save_model']}")


模型已保存 → poem_transformer_v3.pth


## 8. 绘制 Loss 收敛曲线

In [9]:
fig, ax = plt.subplots(figsize=(10, 5))
xs = list(range(1, len(epoch_losses) + 1))
ax.plot(xs, epoch_losses, "b-o", markersize=4, linewidth=1.8, label="Train Loss")
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("Loss",  fontsize=13)
ax.set_title("Training Loss Curve (V3 - Transformer)", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xticks(xs)
fig.tight_layout()
fig.savefig(CONFIG["loss_fig"], dpi=150)
plt.show()
print(f"Loss 曲线 → {CONFIG['loss_fig']}")


Loss 曲线 → training_loss_v3.png


C:\Windows\Temp\ipykernel_40664\495921724.py:12: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 9. 生成函数（三种采样策略）

Transformer 生成与 LSTM 的核心区别：
- LSTM：维护 `hidden state`，每步只输入**当前一个字**
- Transformer：**无状态**，每步把已生成的**完整序列**重新输入，利用因果掩码

| 模式 | 说明 |
|------|------|
| `temperature` | 全量 softmax 采样 |
| `topk` | 只在 top-k 候选中采样 |
| `topp` | nucleus 采样，累积概率截断 |


In [10]:
PUNCT_MAP  = {7: "，", 15: "。", 23: "，", 31: "。"}
ALL_PUNCTS = set("，。！？；、,.")


def _sample_from_logit(logit, mode, temperature, top_k, top_p):
    """从单步 logit (V,) 中按指定策略采样，返回 token id（int）"""
    if mode == "topk":
        values, _ = torch.topk(logit, min(top_k, logit.size(-1)))
        logit[logit < values[-1]] = -1e9
        probs = torch.softmax(logit / temperature, dim=-1)
        return torch.multinomial(probs, 1).item()

    elif mode == "topp":
        probs_sorted, sorted_idx = torch.sort(
            torch.softmax(logit / temperature, dim=-1), descending=True)
        cumsum = torch.cumsum(probs_sorted, dim=-1)
        remove = cumsum - probs_sorted > top_p
        probs_sorted[remove] = 0.0
        probs_sorted /= probs_sorted.sum()
        chosen = torch.multinomial(probs_sorted, 1).item()
        return sorted_idx[chosen].item()

    else:  # temperature
        probs = torch.softmax(logit / temperature, dim=-1)
        return torch.multinomial(probs, 1).item()


def generate(model, start_words, char2idx, idx2char, device,
             mode="topp", temperature=0.8, top_k=10, top_p=0.9,
             seq_len=32):
    """
    Transformer 自回归生成一首七言绝句。

    与 LSTM 不同：每步将完整已生成序列输入模型，
    取最后一个位置的 logit 作为下一字的预测。

    Args:
        start_words : 起始字符串，如 "明月"
        mode        : 采样策略 "temperature" / "topk" / "topp"
        seq_len     : 生成总长度（七言绝句固定 32）
    Returns:
        格式化的 4 行诗字符串
    """
    model.eval()
    unk_id   = char2idx[UNK]
    start_id = char2idx[START]
    all_punct_ids = [char2idx[p] for p in ALL_PUNCTS if p in char2idx]

    # 初始序列：[START] + start_words 对应 id
    generated_ids = [start_id] + [char2idx.get(c, unk_id) for c in start_words]
    generated_chars = list(start_words)

    with torch.no_grad():
        while len(generated_chars) < seq_len:
            # 将已生成序列送入 Transformer
            inp = torch.tensor([generated_ids], dtype=torch.long, device=device)
            logits = model(inp)                        # (1, cur_len, V)
            logit  = logits[0, -1].clone()             # 取最后位置的 logit (V,)

            pos = len(generated_chars)

            if pos in PUNCT_MAP:
                next_char = PUNCT_MAP[pos]
            else:
                logit[all_punct_ids] = -1e9
                next_id   = _sample_from_logit(logit, mode, temperature, top_k, top_p)
                next_char = idx2char.get(next_id, UNK)

            generated_chars.append(next_char)
            generated_ids.append(char2idx.get(next_char, unk_id))

    s     = "".join(generated_chars[:seq_len])
    lines = [s[i*8:(i+1)*8] for i in range(4)]
    return "\n".join(lines)

print("生成函数定义完毕。")


生成函数定义完毕。


## 10. 生成古诗

In [11]:
print("=" * 55)
print(f'以「{CONFIG["start_words"]}」为起始词生成七言绝句（{CONFIG["sample_mode"]} 采样）：')
print("=" * 55)

for i in range(5):
    poem = generate(
        model        = model,
        start_words  = CONFIG["start_words"],
        char2idx     = char2idx,
        idx2char     = idx2char,
        device       = DEVICE,
        mode         = CONFIG["sample_mode"],
        temperature  = CONFIG["temperature"],
        top_k        = CONFIG["top_k"],
        top_p        = CONFIG["top_p"],
    )
    print(f"\n【第 {i+1} 首】\n{poem}")

print("\n" + "=" * 55)


以「明月」为起始词生成七言绝句（topp 采样）：

【第 1 首】
明月方來養物華，
煩君消息又銜花。
當時自有君王在，
曾是當時王母家。

【第 2 首】
明月相催無此情，
去年誰愛別離情。
今朝已是秋聲夜，
却憶江南二月生。

【第 3 首】
明月燈前共一杯，
夜深明月自徘徊。
不知何處消磨盡，
曾見東風又滿來。

【第 4 首】
明月無聲自起遲，
茅簷一點冷參差。
早知風露飄零後，
誰謂先生不是時。

【第 5 首】
明月清風半掩扉，
金陵天上盡芳菲。
隋家宮闕無人到，
落盡楊花空惘歸。



## 11. 三种采样策略对比

In [12]:
modes = [
    ("temperature", dict(temperature=0.8)),
    ("topk",        dict(temperature=0.8, top_k=10)),
    ("topp",        dict(temperature=0.8, top_p=0.9)),
]

for mode_name, kwargs in modes:
    print(f"\n{'─'*45}")
    print(f"  采样策略: {mode_name}   参数: {kwargs}")
    print('─'*45)
    poem = generate(model, CONFIG["start_words"],
                    char2idx, idx2char, DEVICE,
                    mode=mode_name, **kwargs)
    print(poem)



─────────────────────────────────────────────
  采样策略: temperature   参数: {'temperature': 0.8}
─────────────────────────────────────────────
明月離情似一雙，
此身無處可堪降。
飛來欲過無消息，
入海縱橫亦未量。

─────────────────────────────────────────────
  采样策略: topk   参数: {'temperature': 0.8, 'top_k': 10}
─────────────────────────────────────────────
明月樓臺不見時，
玉環樓上玉欄垂。
玉樓春盡無人見，
一曲琵琶一曲詞。

─────────────────────────────────────────────
  采样策略: topp   参数: {'temperature': 0.8, 'top_p': 0.9}
─────────────────────────────────────────────
明月追風我豈知，
絕憐大小獨先詩。
他時若問通天意，
高坐還應怕有知。


## 12. V2（LSTM）vs V3（Transformer）对比分析

运行此单元格前，请确保 V2 模型文件 `poem_lstm_v2.pth` 存在。

In [13]:
import sys, os

v2_path = "poem_lstm_v2.pth"

if not os.path.exists(v2_path):
    print(f"未找到 {v2_path}，跳过对比。请先运行 V2 Notebook 保存模型。")
else:
    # ── 加载 V2 模型 ─────────────────────────────────
    # 需要先定义 ImprovedPoemLSTM（或从 V2 Notebook 复制过来）
    # 这里仅做 Loss 数值对比提示
    ckpt_v2 = torch.load(v2_path, map_location="cpu")
    print("V2 训练配置：")
    cfg2 = ckpt_v2.get("config", {})
    print(f"  hidden_size={cfg2.get('hidden_size')}, "
          f"num_layers={cfg2.get('num_layers')}, "
          f"epochs={cfg2.get('num_epochs')}")

print("\n─── 对比维度 ───────────────────────────────────")
print(f"{'模型':<12} {'参数量':>12} {'采样策略':<12} {'序列建模方式'}")
print(f"{'V2 LSTM':<12} {'（见V2输出）':>12} {'topp':<12} {'循环，有状态'}")
n_v3 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'V3 Transformer':<12} {n_v3:>12,} {'topp':<12} {'并行，因果掩码'}")
print("\n建议：在相同数据和 epoch 下，比较两者最终 Loss 和生成诗句质量。")


V2 训练配置：
  hidden_size=512, num_layers=3, epochs=30

─── 对比维度 ───────────────────────────────────
模型                    参数量 采样策略         序列建模方式
V2 LSTM           （见V2输出） topp         循环，有状态
V3 Transformer    6,335,984 topp         并行，因果掩码

建议：在相同数据和 epoch 下，比较两者最终 Loss 和生成诗句质量。


## 13. （可选）加载已保存模型重新生成

In [14]:
# checkpoint = torch.load(CONFIG["save_model"], map_location=DEVICE)
#
# char2idx = checkpoint["char2idx"]
# idx2char  = checkpoint["idx2char"]
# vocab     = checkpoint["vocab"]
# cfg       = checkpoint["config"]
#
# model2 = PoemTransformer(
#     vocab_size = len(vocab),
#     d_model    = cfg["d_model"],
#     nhead      = cfg["nhead"],
#     num_layers = cfg["num_layers"],
#     d_ffn      = cfg["d_ffn"],
#     dropout    = cfg["dropout"],
#     max_len    = cfg["seq_len"] + 2,
#     activation = cfg["activation"],
# ).to(DEVICE)
# model2.load_state_dict(checkpoint["model_state_dict"])
#
# poem = generate(model2, "明月", char2idx, idx2char, DEVICE, mode="topp")
# print(poem)
